In [1]:
import os
import re
import math
import random
import copy
import time
from datetime import datetime

In [3]:



#евклидово растояние
def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

#стоимость одного маршрута
def route_cost(route, customers):
    cost = 0.0
    for i in range(len(route) - 1):
        cost += euclidean_distance(customers[route[i]], customers[route[i+1]])
    return cost

#суммарная стоимость
def total_cost(solution, instance):
    return sum(route_cost(route, instance['customers']) for route in solution)

#проверка что спрос не превышает грузоподъемность
def is_feasible(solution, instance):
    vehicle_capacity = instance['vehicle_capacity']
    for route in solution:
        load = sum(instance['demands'][i] for i in route if i != 0)
        if load > vehicle_capacity:
            return False
    return True

#инициализация решения, клиенты добавл по очереди до исчерпания лимита
def initial_solution(instance):
    routes = []
    vehicle_capacity = instance['vehicle_capacity']
    customers = list(range(1, len(instance['customers'])))
    current_route = []
    current_load = 0
    for customer in customers:
        demand = instance['demands'][customer]
        if current_load + demand <= vehicle_capacity:
            current_route.append(customer)
            current_load += demand
        else:
            routes.append([0] + current_route + [0])
            current_route = [customer]
            current_load = demand
    if current_route:
        routes.append([0] + current_route + [0])
    return routes

def get_neighbor(solution, instance):
    """
    Используются три оператора:
      - swap: обмен двух клиентов внутри одного маршрута,
      - relocate: перенос клиента из одного маршрута в другой,
      - 2-opt: инверсия последовательности клиентов в маршруте.
    Если новое решение не удовлетворяет ограничениям, возвращается исходное.
    """
    new_solution = copy.deepcopy(solution)
    operator = random.choice(["swap", "relocate", "2opt"])

    if operator == "swap":
        route_idx = random.randint(0, len(new_solution) - 1)
        route = new_solution[route_idx]
        if len(route) > 3:
            idx1, idx2 = random.sample(range(1, len(route) - 1), 2)
            route[idx1], route[idx2] = route[idx2], route[idx1]
        else:
            return new_solution

    elif operator == "relocate":
        source_route_idx = random.randint(0, len(new_solution) - 1)
        source_route = new_solution[source_route_idx]
        if len(source_route) <= 3:
            return new_solution
        cust_idx = random.randint(1, len(source_route) - 2)
        customer = source_route.pop(cust_idx)
        target_route_idx = random.randint(0, len(new_solution) - 1)
        target_route = new_solution[target_route_idx]
        insert_idx = random.randint(1, len(target_route) - 1)
        target_route.insert(insert_idx, customer)

    elif operator == "2opt":
        route_idx = random.randint(0, len(new_solution) - 1)
        route = new_solution[route_idx]
        if len(route) > 4:
            i = random.randint(1, len(route) - 3)
            j = random.randint(i + 1, len(route) - 2)
            route[i:j+1] = reversed(route[i:j+1])
        else:
            return new_solution

    if is_feasible(new_solution, instance):
        return new_solution
    else:
        return solution

def simulated_annealing(instance, initial_temp=9500, final_temp=1e-3, alpha=0.9999, max_iter=300000):
    """
    Имитация отжига с подобранными гиперпараметрами
    Параметры:
      initial_temp - начальная температура,
      final_temp - конечная температура,
      alpha - коэффициент охлаждения,
      max_iter - максимальное число итераций.
    """
    current_solution = initial_solution(instance)
    best_solution = current_solution
    current_cost = total_cost(current_solution, instance)
    best_cost = current_cost
    T = initial_temp
    iteration = 0

    while T > final_temp and iteration < max_iter:
        neighbor = get_neighbor(current_solution, instance)
        neighbor_cost = total_cost(neighbor, instance)
        delta = neighbor_cost - current_cost

        if delta < 0 or random.random() < math.exp(-delta / T):
            current_solution = neighbor
            current_cost = neighbor_cost
            if current_cost < best_cost:
                best_solution = current_solution
                best_cost = current_cost

        T *= alpha
        iteration += 1

    return best_solution, best_cost



In [ ]:
def parse_vrp(file_path):
    """
    Парсинг  .vrp
      NODE_COORD_SECTION - координаты,
      DEMAND_SECTION - спрос,
      CAPACITY - грузоподъемность,
      DEPOT_SECTION - депо.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    customers = []
    demands = []
    vehicle_capacity = None
    is_node_section = False
    is_demand_section = False

    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0] == "CAPACITY":
            vehicle_capacity = int(parts[-1])
        elif parts[0] == "NODE_COORD_SECTION":
            is_node_section = True
            continue
        elif parts[0] == "DEMAND_SECTION":
            is_node_section = False
            is_demand_section = True
            continue
        elif parts[0] == "DEPOT_SECTION":
            break

        if is_node_section:
            customers.append((int(parts[1]), int(parts[2])))
        elif is_demand_section:
            demands.append(int(parts[1]))

    if customers:
        customers.insert(0, customers.pop(0))
    if demands:
        demands.insert(0, 0)

    return {
        'customers': customers,
        'demands': demands,
        'vehicle_capacity': vehicle_capacity
    }

def parse_solution(file_path):
    """
    Парсинг .sol
    Извлекается значение стоимости из строки, начинающейся с "Cost".
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    cost = None
    for line in lines:
        if line.startswith("Cost"):
            cost = int(re.findall(r"\d+", line)[0])
            break
    return cost





# Список папок
folders = ["./A", "./P", "./M"]

results_lines = []
results_lines.append("Имя экземпляра | Найденная стоимость | Оптимальная стоимость | Абсолютное отклонение | % отклонения | Время (сек)\n")
results_lines.append("-" * 120 + "\n")

# Глоб сумм показатели
global_absolute_deviation = 0.0
global_percentage_deviation = 0.0
global_time = 0.0
global_instances = 0

# Для каждого набора отдельные показатели
for folder in folders:
    if not os.path.exists(folder):
        print(f"Папка {folder} не найдена")
        continue

    results_lines.append(f"\n--- Результаты для набора {folder} ---\n")
    folder_absolute_deviation = 0.0
    folder_percentage_deviation = 0.0
    folder_time = 0.0
    folder_instances = 0

    vrp_files = [f for f in os.listdir(folder) if f.endswith(".vrp")]
    for vrp_file in vrp_files:
        instance_name = vrp_file[:-4]  
        vrp_path = os.path.join(folder, vrp_file)
        sol_path = os.path.join(folder, f"{instance_name}.sol")
        
        if not os.path.exists(sol_path):
            print(f"Нет файла оптимального решения для {vrp_file}")
            continue

        # Парсинг 
        instance = parse_vrp(vrp_path)
        optimal_cost = parse_solution(sol_path)
        if optimal_cost is None:
            print(f"Не удалось извлечь оптимальное решение для {vrp_file}")
            continue

        #время
        start_time = time.time()
        best_solution, best_cost = simulated_annealing(instance)
        elapsed_time = time.time() - start_time

        # Отклонение
        absolute_deviation = best_cost - optimal_cost
        percentage_deviation = (absolute_deviation / optimal_cost) * 100 if optimal_cost != 0 else 0

        # Обновляем глобальные и локальные суммарные показатели
        global_absolute_deviation += absolute_deviation
        global_percentage_deviation += percentage_deviation
        global_time += elapsed_time
        global_instances += 1

        folder_absolute_deviation += absolute_deviation
        folder_percentage_deviation += percentage_deviation
        folder_time += elapsed_time
        folder_instances += 1

        result_line = f"{instance_name:15} | {best_cost:20.2f} | {optimal_cost:20} | {absolute_deviation:20.2f} | {percentage_deviation:10.2f}% | {elapsed_time:10.2f}\n"
        results_lines.append(result_line)
        results_lines.append("-" * 120 + "\n")
        print(f"Обработан {instance_name} из папки {folder}")

    # Подсчёт средних значений
    if folder_instances > 0:
        avg_folder_absolute_deviation = folder_absolute_deviation / folder_instances
        avg_folder_percentage_deviation = folder_percentage_deviation / folder_instances
        avg_folder_time = folder_time / folder_instances
    else:
        avg_folder_absolute_deviation = 0
        avg_folder_percentage_deviation = 0
        avg_folder_time = 0

    results_lines.append(f"Набор {folder} - экземпляров: {folder_instances}\n")
    results_lines.append(f"Сред абс отклонение: {avg_folder_absolute_deviation:.2f}\n")
    results_lines.append(f"Сред % отклонение: {avg_folder_percentage_deviation:.2f}%\n")
    results_lines.append(f"Сред время выполнения: {avg_folder_time:.2f} сек\n")
    results_lines.append("\n" + "=" * 120 + "\n\n")

# Подсчёт глобальных средних значений
if global_instances > 0:
    avg_absolute_deviation = global_absolute_deviation / global_instances
    avg_percentage_deviation = global_percentage_deviation / global_instances
    avg_time = global_time / global_instances
else:
    avg_absolute_deviation = 0
    avg_percentage_deviation = 0
    avg_time = 0

results_lines.append("\n--- Итоговые результаты по всем наборам ---\n")
results_lines.append(f"Общ количество экземпляров: {global_instances}\n")
results_lines.append(f"Сред абсолютное отклонение: {avg_absolute_deviation:.2f}\n")
results_lines.append(f"Сред % отклонение: {avg_percentage_deviation:.2f}%\n")
results_lines.append(f"Сред время выполнения: {avg_time:.2f} сек\n")




Обработан A-n32-k5 из папки ./A
Обработан A-n33-k5 из папки ./A
Обработан A-n33-k6 из папки ./A
Обработан A-n34-k5 из папки ./A
Обработан A-n36-k5 из папки ./A
Обработан A-n37-k5 из папки ./A
Обработан A-n37-k6 из папки ./A
Обработан A-n38-k5 из папки ./A


In [ ]:
#сохраняю в results
os.makedirs("results", exist_ok=True)


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result_filename = os.path.join("results", f"results_{timestamp}.txt")

with open(result_filename, "w", encoding="utf-8") as f:
    f.writelines(results_lines)

print(f"\nРезультаты записаны в файл {result_filename}")